In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/Group project')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

from Discriminator import Discriminator
from generator import Generator                                                             #######  <---------------

# Training Configuration
LATENT_DIM = 100
BATCH_SIZE = 100
NUM_EPOCHS = 100
LEARNING_RATE = 0.0002
BETA1 = 0.5
BETA2 = 0.999

# Paths
DATA_DIR = '/content/drive/MyDrive/Group project'
OUTPUT_DIR = '/content/drive/MyDrive/Group project/gan_output'
CHECKPOINT_DIR = '/content/drive/MyDrive/Group project/checkpoints'

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


def load_preprocessed_data(data_dir):
    """
    Load preprocessed ECG data from .npy file.

    Returns:
        DataLoader with batches of shape (batch_size, 3000, 1)
    """
    print("Loading preprocessed data...")
    data_path = os.path.join(data_dir, 'ecg_train.npy')

    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Data file not found: {data_path}")

    # Load data
    data = np.load(data_path)
    print(f"Loaded data shape: {data.shape}")

    # Convert to tensor and add channel dimension
    data_tensor = torch.FloatTensor(data).unsqueeze(-1)  # (N, 3000, 1)

    # Create dataset and dataloader
    dataset = TensorDataset(data_tensor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    print(f"Created DataLoader with {len(dataloader)} batches")
    return dataloader


def initialize_weights(model):
    """Initialize model weights using normal distribution."""
    for m in model.modules():
        if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
            nn.init.normal_(m.weight.data, 0.0, 0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias.data, 0)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)


def train_gan(dataloader, num_epochs=NUM_EPOCHS):
    """
    Train GAN with discriminator and generator.

    Training loop:
    1. Train Discriminator on real data (from preprocessing)
    2. Train Discriminator on fake data (from generator)
    3. Train Generator to fool discriminator
    """

    # Initialize models
    print("\nInitializing models...")
    generator = Generator(latent_dim=LATENT_DIM, seq_length=3000).to(device)
    discriminator = Discriminator(seq_length=3000, dropout=0.3).to(device)

    # Initialize weights
    initialize_weights(generator)                                                             #######  <---------------
    initialize_weights(discriminator)

    # Optimizers
    optimizer_G = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))    #######  <---------------
    optimizer_D = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))

    # Loss function
    criterion = nn.BCELoss()

    # Create output directories
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # Training history
    history = {
        'g_loss': [],
        'd_loss': [],
        'd_real_acc': [],
        'd_fake_acc': []
    }

    print(f"\nStarting training for {num_epochs} epochs...")
    print(f"Batches per epoch: {len(dataloader)}")
    print("="*60)

    for epoch in range(num_epochs):
        g_losses = []
        d_losses = []
        d_real_accs = []
        d_fake_accs = []

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for i, (real_data,) in enumerate(progress_bar):
            batch_size = real_data.size(0)
            real_data = real_data.to(device)  # (batch_size, 3000, 1) - Real data from preprocessing

            # Labels
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # ==========================================
            # Train Discriminator
            # ==========================================
            optimizer_D.zero_grad()

            # 1. Train on REAL data (from preprocessing output)
            real_pred = discriminator(real_data)
            d_real_loss = criterion(real_pred, real_labels)

            # 2. Train on FAKE data (from generator output)
            noise = torch.randn(batch_size, LATENT_DIM).to(device)
            fake_data = generator(noise)  # (batch_size, 3000, 1) - Fake data from generator   #######  <---------------
            fake_pred = discriminator(fake_data.detach())
            d_fake_loss = criterion(fake_pred, fake_labels)

            # Total discriminator loss
            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            optimizer_D.step()

            # ==========================================
            # Train Generator
            # ==========================================
            optimizer_G.zero_grad()

            # Generate fake data and try to fool discriminator
            noise = torch.randn(batch_size, LATENT_DIM).to(device)
            fake_data = generator(noise)                                            #######  <---------------
            fake_pred = discriminator(fake_data)

            # Generator wants discriminator to classify fake as real
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            optimizer_G.step()

            # ==========================================
            # Track metrics
            # ==========================================
            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

            # Calculate accuracy
            d_real_acc = (real_pred > 0.5).float().mean().item()
            d_fake_acc = (fake_pred < 0.5).float().mean().item()
            d_real_accs.append(d_real_acc)
            d_fake_accs.append(d_fake_acc)

            # Update progress bar
            progress_bar.set_postfix({
                'D_loss': f'{d_loss.item():.4f}',
                'G_loss': f'{g_loss.item():.4f}',
                'D_real': f'{d_real_acc:.2f}',
                'D_fake': f'{d_fake_acc:.2f}'
            })

        # Epoch statistics
        avg_g_loss = np.mean(g_losses)
        avg_d_loss = np.mean(d_losses)
        avg_d_real_acc = np.mean(d_real_accs)
        avg_d_fake_acc = np.mean(d_fake_accs)

        history['g_loss'].append(avg_g_loss)
        history['d_loss'].append(avg_d_loss)
        history['d_real_acc'].append(avg_d_real_acc)
        history['d_fake_acc'].append(avg_d_fake_acc)

        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"  G_loss: {avg_g_loss:.4f} | D_loss: {avg_d_loss:.4f}")
        print(f"  D_real_acc: {avg_d_real_acc:.2f} | D_fake_acc: {avg_d_fake_acc:.2f}")

        # Save generated samples every 10 epochs
        if (epoch + 1) % 10 == 0:
            save_generated_samples(generator, epoch + 1)        #######  <---------------
            save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, epoch + 1) #######  <---------------

    # Save final models
    save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, num_epochs, final=True)
    plot_training_history(history)

    print("\n" + "="*60)
    print("Training complete!")
    print(f"Models saved to: {CHECKPOINT_DIR}")
    print(f"Generated samples saved to: {OUTPUT_DIR}")
    print("="*60)

    return generator, discriminator, history                                    #######  <---------------


def save_generated_samples(generator, epoch, num_samples=5):                               #######  <---------------
    """Save generated ECG samples as images."""
    generator.eval()                                                              #######  <---------------
    with torch.no_grad():
        noise = torch.randn(num_samples, LATENT_DIM).to(device)
        fake_data = generator(noise).cpu().numpy()                                #######  <---------------

    fig, axes = plt.subplots(num_samples, 1, figsize=(15, 2*num_samples))
    for i in range(num_samples):
        axes[i].plot(fake_data[i, :, 0])
        axes[i].set_title(f"Generated ECG Sample {i+1}")
        axes[i].set_xlabel("Time Steps")
        axes[i].set_ylabel("Amplitude")
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'generated_epoch_{epoch}.png'))
    plt.close()

    generator.train()                                                           #######  <---------------


def save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, epoch, final=False):#######  <---------------
    """Save model checkpoints."""
    suffix = 'final' if final else f'epoch_{epoch}'

    checkpoint = {
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),                               #######  <---------------
        'discriminator_state_dict': discriminator.state_dict(),
        'optimizer_G_state_dict': optimizer_G.state_dict(),
        'optimizer_D_state_dict': optimizer_D.state_dict(),
    }

    torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, f'checkpoint_{suffix}.pth'))


def plot_training_history(history):
    """Plot training loss and accuracy curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss curves
    ax1.plot(history['g_loss'], label='Generator Loss')                               #######  <---------------
    ax1.plot(history['d_loss'], label='Discriminator Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy curves
    ax2.plot(history['d_real_acc'], label='D Real Accuracy')
    ax2.plot(history['d_fake_acc'], label='D Fake Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Discriminator Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'))
    plt.close()


if __name__ == '__main__':
    # Load preprocessed data
    dataloader = load_preprocessed_data(DATA_DIR)

    # Train GAN
    generator, discriminator, history = train_gan(dataloader, num_epochs=NUM_EPOCHS)#######  <---------------

    print("\n✓ Training complete!")

Using device: cpu
Loading preprocessed data...
Loaded data shape: (21021, 3000)
Created DataLoader with 210 batches

Initializing models...

Starting training for 100 epochs...
Batches per epoch: 210


Epoch 1/100:  16%|█▌        | 33/210 [54:07<4:50:17, 98.40s/it, D_loss=4.1553, G_loss=6.2838, D_real=0.05, D_fake=0.94]


KeyboardInterrupt: 